# Anti Money Laudenring Study
---

The primary objective of this analysis is to develop a machine learning model capable of identifying potentially suspicious transactions or customers for AML purposes.

We aim to perform data preprocessing and feature engineering to create meaningful inputs for modeling and then train and evaluate a machine learning model to detect suspicious behavior. Finally, we will provide interpretable insights that can support AML analysts and compliance decision-making.

Follow bellow some references for this study:

#### Clustering 

[1] Chandola, Varun, Arindam Banerjee, and Vipin Kumar. "Anomaly detection: A survey." ACM computing surveys (CSUR) 41.3 (2009): 1-58. Access on: https://dl.acm.org/doi/abs/10.1145/1541880.1541882

[2] Knox, Edwin M., and Raymond T. Ng. "Algorithms for mining distancebased outliers in large datasets." Proceedings of the international conference on very large data bases. Citeseer, 1998. Access on: https://www.researchgate.net/publication/2346156_Algorithms_for_Mining_Distance-Based_Outliers_in_Large_Datasets

[3] Jensen, Rasmus Ingemann Tuffveson, and Alexandros Iosifidis. "Fighting money laundering with statistics and machine learning." Ieee Access 11 (2023): 8889-8903. Access on: https://ieeexplore.ieee.org/stamp/stamp.jsp?arnumber=10025710

#### Dimension reduction

[4] Bakry, Ahmed N., et al. "Combating financial crimes with unsupervised learning techniques: clustering and dimensionality reduction for anti-money laundering." arXiv preprint arXiv:2403.00777 (2024). Access on: https://arxiv.org/pdf/2403.00777

#### System architecture
[5] Sizan, Mir Mohtasam Hossain. "Machine learning-based unsupervised ensemble approach for detecting new money laundering typologies in transaction graphs." International Journal of Applied Mathematics 38.2s (2025): 351-374.

### Imports
---

In [ ]:
# system
import os
import sys
ROOT_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(ROOT_PATH)

# utils
from tqdm import tqdm

#visualization
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
%matplotlib inline

# data manipulation
import pandas as pd
import numpy as np
random_state = 42
np.random.seed(random_state)
pd.options.display.float_format = '{:,.2f}'.format
pd.set_option('display.max_columns', None)

#machine learning
from sklearn.ensemble import IsolationForest, RandomForestRegressor
import shap
from scipy.spatial.distance import mahalanobis

#cluster
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.manifold import TSNE
from sklearn.manifold import trustworthiness
#from sklearn.cluster import HDBSCAN
from hdbscan import HDBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score, r2_score

#scalers
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler


# personalized modules
from src.utils.dataset import get_raw_transactions

### Load data
---

To get started, we are going to load data from IBM dataset available on Kaggle. You can get the data on the following link: https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml.
After downloading it, make sure that the data is available in the right path "src/data/HI-Small_Trans.csv"

In [ ]:
data_path = f"{ROOT_PATH}\\src\\data\\preprocessed_customer_level_features.csv"
feature_df = get_raw_transactions(data_path)

In [ ]:
feature_df.head()

In [ ]:
feature_df.shape

### Anti Money Laundering Architecture
---

#### Handling skewed data
After analyzing the engineered customer features, it is possible to observe that the data is highly skewed. Most customers exhibit similar transactional behavior, while a small number present significantly higher activity levels, resulting in a pronounced right-tailed distribution.

To address this issue, we apply a logarithmic transformation to the affected features. Since the skewness is concentrated in the right tail, the log transformation helps compress extreme values, reduce the influence of outliers, and produce a more stable distribution for modeling.

This transformation improves model robustness and allows variance-based techniques to perform more effectively.

For more details of how this approach means, refers to the following documentation: https://blog.gopenai.com/how-to-handle-skewed-data-a-guide-for-data-scientists-84187ba7f805

In [ ]:
clustering_df = feature_df.copy()#.sample(n=int(10e3), random_state=random_state)

In [ ]:
cluster_columns = list(clustering_df.columns)
cluster_columns.remove('customer_id')

In [ ]:
cols = cluster_columns
n_cols = 4
n_rows = int(np.ceil(len(cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(cols):
    data = clustering_df[col]    
    axes[i].hist(data, bins=30)
    axes[i].set_title(col)
    axes[i].tick_params(axis='x', rotation=45)
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
skew = clustering_df[cluster_columns].skew()

In [ ]:
skew

In [ ]:
log_transformed_features_df = clustering_df.copy()
skewed_cols = skew[skew > 1].index
for col in skewed_cols:
    log_transformed_features_df[col] = np.log1p(log_transformed_features_df[col])

In [ ]:
cols = cluster_columns
n_cols = 4
n_rows = int(np.ceil(len(cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(cols):
    data = log_transformed_features_df[col]    
    axes[i].hist(data, bins=30)
    axes[i].set_title(col)
    axes[i].tick_params(axis='x', rotation=45)
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Feature Distributions (Log Applied When Skewed)", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
#raw data
X_raw = clustering_df[cluster_columns]

#scaled data
X_robust_scaled = RobustScaler().fit_transform(X_raw)
X_standard_scaled = StandardScaler().fit_transform(X_raw)

#log transformed data
X_log_transformed = log_transformed_features_df[cluster_columns]
X_sample_log_tranformed = X_log_transformed.sample(n=int(25e3), random_state=random_state)

X_log_standard_scaled = StandardScaler().fit_transform(X_log_transformed)
X_log_robust_scaled = RobustScaler().fit_transform(X_log_transformed)

#sample scaled data
X_sample_log_robust_scaled = RobustScaler().fit_transform(X_sample_log_tranformed)
X_sample_log_standard_scaled = StandardScaler().fit_transform(X_sample_log_tranformed)

#### Dimension Reduction
To visualize customer groupings, we apply a dimensionality reduction technique, as the original dataset is high-dimensional and cannot be directly plotted. Reducing the number of dimensions allows us to project the data into a two-dimensional space for exploratory analysis.

We begin with Principal Component Analysis (PCA), a linear technique that transforms the original features into uncorrelated components while preserving as much variance as possible. The first two principal components are used to create scatter plots that help assess clustering structure and separation.

In [ ]:
pca = PCA(n_components=6, random_state=random_state)
X_pca = pca.fit_transform(X_log_standard_scaled)
explained_variance = np.cumsum(pca.explained_variance_ratio_)

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(range(1, len(explained_variance) + 1)),
    y=explained_variance,
    mode='lines+markers'
))

fig.update_layout(
    title="Cumulative Explained Variance by PCA Components",
    xaxis_title="Number of Components",
    yaxis_title="Cumulative Explained Variance",
    yaxis_tickformat=".0%",
)

fig.show()

In [ ]:
pca_df = pd.DataFrame(
    {
        'x': X_pca[:, 0],
        'y': X_pca[:, 1]
    }
)

In [ ]:
sample_size = n=int(25e3)
pca_df_sample = pca_df.sample(sample_size, random_state=random_state)

In [ ]:
fig = px.scatter(
    pca_df_sample,
    x='x',
    y='y',
    opacity=0.1,
    title="Customer Behavior Distribution - PCA",
)
fig.update_traces(marker=dict(color="blue"))
fig.show()

In [ ]:
print(f"Sampling {sample_size} data points for t-SNE visualization...")

idx = np.random.choice(X_log_standard_scaled.shape[0], size=sample_size, replace=False)
X_sample = X_log_standard_scaled[idx]

In [ ]:
tsne = TSNE(n_components=2, random_state=random_state, perplexity=25, n_iter_without_progress=300)
X_tsne = tsne.fit_transform(X_sample)

In [ ]:
tsne_df = pd.DataFrame(
    {
        'x': X_tsne[:, 0],
        'y': X_tsne[:, 1]
    }
)

In [ ]:
fig = px.scatter(
    tsne_df,
    x='x',
    y='y',
    opacity=0.1,
    title="Customer Behavior Distribution - TSNE"
)
fig.update_layout(height=600)
fig.update_traces(marker=dict(color="blue"))
fig.show()

In [ ]:
tsne_score = trustworthiness(X_sample, X_tsne, n_neighbors=5)
print("TSNE Trustworthiness:", round(tsne_score, 4))

#### Clustering with K-Means

In [ ]:
k_results = []
g_results = []
k_range = range(2, 10)
for k in tqdm(k_range, desc="Evaluating K-means"):
    kmeans = KMeans(n_clusters=k, random_state=random_state)
    gmm = GaussianMixture(n_components=k, covariance_type='diag', random_state=random_state)

    k_labels = kmeans.fit_predict(X_log_standard_scaled)
    g_labels = gmm.fit_predict(X_log_standard_scaled)

    si_score = silhouette_score(X_log_standard_scaled, k_labels, sample_size = sample_size, random_state=random_state)
    ch_score = calinski_harabasz_score(X_log_standard_scaled, k_labels)
    db_score = davies_bouldin_score(X_log_standard_scaled, k_labels)

    gmm_si_score = silhouette_score(X_log_standard_scaled, g_labels, sample_size = sample_size, random_state=random_state)
    gmm_ch_score = calinski_harabasz_score(X_log_standard_scaled, g_labels)
    gmm_db_score = davies_bouldin_score(X_log_standard_scaled, g_labels)

    k_results.append((k, si_score, ch_score, db_score))
    g_results.append((k, gmm_si_score, gmm_ch_score, gmm_db_score))

In [ ]:
k_results = np.array(k_results)
g_results = np.array(g_results)

k_values = k_results[:, 0]

k_silhouette_scores = k_results[:, 1]
k_calinski_harabasz_scores = k_results[:, 2]
k_davies_bouldin_scores = k_results[:, 3]

g_silhouette_scores = g_results[:, 1]
g_calinski_harabasz_scores = g_results[:, 2]
g_davies_bouldin_scores = g_results[:, 3]

In [ ]:
# Define consistent colors
KMEANS_COLOR = "#1f77b4"   # blue
GMM_COLOR = "#d62728"      # red

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Silhouette Score (Higher is Better)",
        "Calinski-Harabasz Score (Higher is Better)",
        "Davies-Bouldin Score (Lower is Better)",
        ""
    )
)

best_sil_k = k_values[np.argmax(k_silhouette_scores)]
best_ch_k = k_values[np.argmax(k_calinski_harabasz_scores)]
best_db_k = k_values[np.argmin(k_davies_bouldin_scores)]

# ---------- Silhouette ----------
fig.add_trace(
    go.Scatter(
        x=k_values, y=k_silhouette_scores,
        mode='lines',
        name="KMeans",
        line=dict(width=3, color=KMEANS_COLOR),
        marker=dict(size=8, color=KMEANS_COLOR)
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=k_values, y=g_silhouette_scores,
        mode='lines',
        name="GMM",
        line=dict(width=3, color=GMM_COLOR),
        marker=dict(size=8, color=GMM_COLOR)
    ),
    row=1, col=1
)

fig.add_vline(x=best_sil_k, line_dash="dot", line_color="gray", row=1, col=1)

# ---------- CH ----------
fig.add_trace(
    go.Scatter(
        x=k_values, y=k_calinski_harabasz_scores,
        mode='lines',
        line=dict(width=3, color=KMEANS_COLOR),
        marker=dict(size=8, color=KMEANS_COLOR),
        showlegend=False
    ),
    row=1, col=2
)

fig.add_trace(
    go.Scatter(
        x=k_values, y=g_calinski_harabasz_scores,
        mode='lines',
        line=dict(width=3, color=GMM_COLOR),
        marker=dict(size=8, color=GMM_COLOR),
        showlegend=False
    ),
    row=1, col=2
)

fig.add_vline(x=best_ch_k, line_dash="dot", line_color="gray", row=1, col=2)

# ---------- DB ----------
fig.add_trace(
    go.Scatter(
        x=k_values, y=k_davies_bouldin_scores,
        mode='lines',
        line=dict(width=3, color=KMEANS_COLOR),
        marker=dict(size=8, color=KMEANS_COLOR),
        showlegend=False
    ),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(
        x=k_values, y=g_davies_bouldin_scores,
        mode='lines',
        line=dict(width=3, color=GMM_COLOR),
        marker=dict(size=8, color=GMM_COLOR),
        showlegend=False
    ),
    row=2, col=1
)

fig.add_vline(x=best_db_k, line_dash="dot", line_color="gray", row=2, col=1)

fig.update_layout(
    template="plotly_white",
    height=850,
    width=1200,
    title="Clustering Evaluation Metrics vs Number of Clusters",
    legend=dict(
        orientation="h",
        y=1.05,
        x=0.5,
        xanchor="center"
    )
)

fig.update_xaxes(title_text="Number of Clusters (k)")
fig.update_yaxes(title_text="Score")

fig.show()

In [ ]:
kmeans = KMeans(n_clusters=7, random_state=random_state)
kmeans.fit(X_log_standard_scaled)

In [ ]:
clustering_df['kmeans_cluster'] = kmeans.predict(X_log_standard_scaled)

In [ ]:
clustering_df["kmeans_cluster"].value_counts(False)

In [ ]:
X_log_standard_scaled

In [ ]:
X_log_standard_scaled_df = pd.DataFrame(X_log_standard_scaled, columns=cluster_columns)

In [ ]:
progress_bar = tqdm(clustering_df["kmeans_cluster"].unique(), desc="Calculating Mahalanobis Distances")
for cluster in clustering_df["kmeans_cluster"].unique():
    cluster_mask = clustering_df["kmeans_cluster"] == cluster
    cluster_idx = clustering_df[cluster_mask].index

    x_cluster = X_log_standard_scaled[cluster_mask]
    x_cluster_df = pd.DataFrame(x_cluster, columns=cluster_columns, index=cluster_idx)

    mu = np.mean(x_cluster, axis=0)
    cov = np.cov(x_cluster, rowvar=False)
    cov += np.eye(cov.shape[0]) * 1e-6

    cov_inv = np.linalg.pinv(cov)
    
    x_cluster_df["mahalanobis_distance"] = x_cluster_df.apply(lambda row: mahalanobis(row, mu, cov_inv), axis=1)
    x_cluster_df["mahalanobis_risk_score"] = MinMaxScaler(feature_range=(0, 1)).fit_transform(x_cluster_df[["mahalanobis_distance"]])
    
    clustering_df.loc[cluster_mask, "mahalanobis_distance"] = x_cluster_df["mahalanobis_distance"]
    clustering_df.loc[cluster_mask, "mahalanobis_risk_score"] = x_cluster_df["mahalanobis_risk_score"]
    
    progress_bar.update(1)
progress_bar.close()

In [ ]:
pca_df['mahalanobis_risk_score'] = clustering_df['mahalanobis_risk_score']
tsne_df['mahalanobis_risk_score'] = clustering_df.iloc[idx].reset_index(drop=True)['mahalanobis_risk_score']

pca_df['kmeans_cluster'] = clustering_df['kmeans_cluster'].astype(str)
tsne_df['kmeans_cluster'] = clustering_df.iloc[idx].reset_index(drop=True)['kmeans_cluster'].astype(str)

In [ ]:
pca_df_sample = pca_df.sample(sample_size, random_state=random_state)

In [ ]:
fig = px.scatter(
    pca_df_sample,
    x='x',
    y='y',
    color='kmeans_cluster',
    color_discrete_sequence=px.colors.qualitative.Set1,
    opacity=0.05,
    title="PCA – KMeans Clusters"
)
fig.show()

In [ ]:
fig = px.scatter(
    tsne_df,
    x='x',
    y='y',
    color='kmeans_cluster',
    color_discrete_sequence=px.colors.qualitative.Set1,
    opacity=0.2,
    title="TSNE – KMeans Clusters"
)
fig.show()

In [ ]:
fig = px.scatter(
    pca_df_sample,
    x='x',
    y='y',
    color='mahalanobis_risk_score',
    color_continuous_scale=px.colors.sequential.Viridis,
    opacity=0.2,
    title="PCA – Mahalanobis Risk Scores"
)
fig.show()

In [ ]:
fig = px.scatter(
    tsne_df,
    x='x',
    y='y',
    color='mahalanobis_risk_score',
    color_continuous_scale=px.colors.sequential.Viridis,
    opacity=0.1,
    title="TSNE – Mahalanobis Risk Scores"
)
fig.show()

In [ ]:
fig = px.violin(
    clustering_df,
    y="mahalanobis_risk_score",
    box=True,
    points=False,
    title="Mahalanobis - Customer Risk Score Distribution"
)

fig.show()

In [ ]:
clustering_df[clustering_df["kmeans_cluster"]==1].sort_values("mahalanobis_risk_score", ascending=False)

#### Clustering with HDBSCAN

In [ ]:
def recursive_search(X, param_grid, keys, current_params={}, depth=0, results=[]):
    if depth == len(keys):
        clusterer = HDBSCAN(**current_params, gen_min_span_tree=True)
        clusterer.fit(X)
        labels = clusterer.labels_

        noise_ratio = np.mean(labels == -1)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

        validity_score = clusterer.relative_validity_

        results.append({
            **current_params,
            "validity": validity_score,
            "noise_ratio": noise_ratio,
            "n_clusters": n_clusters
        })
        #add progress bar update here
        print(f"Evaluated: {current_params} | Validity: {validity_score:.4f} | Noise Ratio: {noise_ratio:.4f} | Clusters: {n_clusters}")
        return
    
    key = keys[depth]
    for value in param_grid[key]:
        new_params = current_params.copy()
        new_params[key] = value
        recursive_search(X, param_grid, keys, new_params, depth+1, results)

In [ ]:
n = X_log_robust_scaled.shape[0]
param_grid = {
    "min_cluster_size": [100, 500, 1000, 2000],
    "min_samples": [20, 40, 80, 120],
    "metric": ["euclidean", "manhattan"],
    "cluster_selection_method": ["eom", "leaf"]
}

keys = list(param_grid.keys())
results = []
recursive_search(X_sample_log_robust_scaled, param_grid, keys, results=results)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by=["validity"],
    ascending=False
)

results_df.head()

In [ ]:
#create best params dict without the evaluation metrics
best_params = results_df.iloc[0][keys].to_dict()
best_params

In [ ]:
#mock 
best_params = {'min_cluster_size': 1000,
 'min_samples': 120,
 'metric': 'manhattan',
 'cluster_selection_method': 'eom'}

In [ ]:
clusterer = HDBSCAN(**best_params, gen_min_span_tree=True)

In [ ]:
labels = clusterer.fit_predict(X_log_robust_scaled)

In [ ]:
clusterer.relative_validity_

In [ ]:
clustering_df["hdbscan_cluster"] = labels
clustering_df['hdbscan_risk_score'] = clusterer.outlier_scores_
#clustering_df['hdbscan_risk_score'] = clustering_df['hdbscan_risk_score'].fillna(0)
pca_df['hdbscan_risk_score'] = clustering_df['hdbscan_risk_score']
pca_df['hdbscan_cluster'] = clustering_df['hdbscan_cluster']

In [ ]:
tsne_df['hdbscan_cluster'] = clustering_df.iloc[idx].reset_index(drop=True)['hdbscan_cluster']
tsne_df['hdbscan_risk_score'] = clustering_df.iloc[idx].reset_index(drop=True)['hdbscan_risk_score']

In [ ]:
pca_df_sample = pca_df.sample(n=sample_size, random_state=random_state)

In [ ]:
fig = px.violin(
    clustering_df,
    y="hdbscan_risk_score",
    box=False,
    points=False,
    title="HDBSCAN - Customer Risk Score Distribution"
)

fig.show()

fig = px.scatter(
    pca_df_sample,
    x='x',
    y='y',
    color='hdbscan_risk_score',
    color_continuous_scale=px.colors.sequential.Plasma,
    opacity=0.1,
    title="PCA – HDBSCAN Risk Score"
)
fig.update_traces(marker=dict(size=6))
fig.show()

#plot clusters
fig = px.scatter(
    pca_df_sample,
    x='x',
    y='y',
    color=pca_df_sample['hdbscan_cluster'],
    color_continuous_scale=px.colors.sequential.Plasma,
    opacity=0.1,
    title="PCA – HDBSCAN Clusters"
)
fig.show()

#### Isolation Forest

In [ ]:
iso = IsolationForest(
    random_state=random_state,
    contamination=0.05
)

In [ ]:
iso.fit(X_robust_scaled)

In [ ]:
clustering_df['iso_risk_score'] = (-1)*iso.score_samples(X_robust_scaled)
pca_df['iso_risk_score'] = clustering_df['iso_risk_score']

In [ ]:
tsne_df['iso_risk_score'] = clustering_df.iloc[idx].reset_index(drop=True)['iso_risk_score']

In [ ]:
clustering_df['iso_risk_score'].describe()

In [ ]:
pca_df_sample = pca_df.sample(n=sample_size, random_state=random_state)

In [ ]:
fig = px.violin(
    clustering_df,
    y="iso_risk_score",
    box=False,
    points=False,
    title="Isolation Forest – Risk Score Density"
)

fig.show()

fig = px.scatter(
    pca_df_sample,
    x='x',
    y='y',
    color='iso_risk_score',
    color_continuous_scale=px.colors.sequential.Plasma,
    opacity=0.2,
    title="PCA – Isolation Forest Risk Score"
)
fig.update_traces(marker=dict(size=6))
fig.show()


In [ ]:
fig = px.scatter(
    tsne_df,
    x='x',
    y='y',
    color='iso_risk_score',
    color_continuous_scale=px.colors.sequential.Plasma,
    opacity=0.2,
    title="TNSE – Isolation Forest Risk Score"
)
fig.update_traces(marker=dict(size=6))
fig.show()

#### Score summary

In [ ]:
clustering_df.head()

In [ ]:
clustering_df['final_score'] = clustering_df['hdbscan_risk_score']*0.3 + clustering_df['mahalanobis_risk_score']*0.3 + clustering_df['iso_risk_score']*0.4
pca_df['final_score'] = pca_df['hdbscan_risk_score']*0.3 + pca_df['mahalanobis_risk_score']*0.3 + pca_df['iso_risk_score']*0.4

In [ ]:
tsne_df['final_score'] = tsne_df['hdbscan_risk_score']*0.3 + tsne_df['mahalanobis_risk_score']*0.3 + tsne_df['iso_risk_score']*0.4

In [ ]:
fig = px.violin(
    clustering_df,
    y="final_score",
    box=False,
    points=False,
    title="Combined Risk Score Density"
)
fig.show()

In [ ]:
pca_df_sample = pca_df.sample(n=sample_size, random_state=random_state)

In [ ]:
fig = px.scatter(
    pca_df_sample,
    x='x',
    y='y',
    color='hdbscan_cluster',
    color_continuous_scale=px.colors.sequential.Plasma,
    opacity=0.2,
    title="PCA – HDBSCAN Cluster Distribution"
)
fig.update_traces(marker=dict(size=6))
fig.show()

In [ ]:
fig = px.scatter(
    pca_df_sample,
    x='x',
    y='y',
    color='hdbscan_risk_score',
    color_continuous_scale=px.colors.sequential.Plasma,
    opacity=0.2,
    title="PCA – HDBSCAN Risk Score"
)
fig.update_traces(marker=dict(size=6))
fig.show()

In [ ]:
fig = px.scatter(
    pca_df_sample,
    x='x',
    y='y',
    color='iso_risk_score',
    color_continuous_scale=px.colors.sequential.Plasma,
    opacity=0.2,
    title="PCA – Isolation Forest Risk Score"
)
fig.update_traces(marker=dict(size=6))
fig.show()

In [ ]:
fig = px.scatter(
    pca_df_sample,
    x='x',
    y='y',
    color='mahalanobis_risk_score',
    color_continuous_scale=px.colors.sequential.Plasma,
    opacity=0.2,
    title="PCA – Mahalanobis Risk Score"
)
fig.update_traces(marker=dict(size=6))
fig.show()

In [ ]:
fig = px.scatter(
    pca_df_sample,
    x='x',
    y='y',
    color='final_score',
    color_continuous_scale=px.colors.sequential.Plasma,
    opacity=0.2,
    title="PCA – Combined Risk Score"
)
fig.update_traces(marker=dict(size=6))
fig.show()

In [ ]:
fig = px.scatter(
    tsne_df,
    x='x',
    y='y',
    color='final_score',
    color_continuous_scale=px.colors.sequential.Plasma,
    opacity=0.2,
    title="t-SNE – Combined Risk Score"
)
fig.update_traces(marker=dict(size=6))
fig.show()

In [ ]:
reduced_clusters = clustering_df[clustering_df["final_score"] > 0.5].shape[0]

In [ ]:
clustering_df[clustering_df["final_score"] > 0.5].sort_values("final_score", ascending=False)

In [ ]:
print(f"Number of total customers: {clustering_df.shape[0]}")
print(f"Number of reduced clusters: {reduced_clusters}")
print(f"Percentage of customers flagged as high risk: {reduced_clusters/clustering_df.shape[0]*100:.2f}%")

In [ ]:
clustering_df.to_csv(f"{ROOT_PATH}\\src\\data\\analysis\\customer_level_risk_score.csv", index=False)